In [3]:

import requests, re, json
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"}

def scrape_product(url):
    """Pobierz pełne dane produktu ze strony bechcicki.pl"""
    r = requests.get(url, headers=HEADERS, timeout=20)
    soup = BeautifulSoup(r.text, 'html.parser')
    data = {'url': url}
    
    # 1. JSON-LD — nazwa, opis, zdjęcia, EAN, SKU, marka
    for script in soup.find_all('script', type='application/ld+json'):
        try:
            obj = json.loads(script.string or '{}')
            if isinstance(obj, list): obj = next((o for o in obj if o.get('@type')=='Product'), {})
            if obj.get('@type') == 'Product':
                data['name']  = obj.get('name','')
                data['desc']  = obj.get('description','')
                data['ean']   = obj.get('gtin13') or obj.get('gtin') or obj.get('gtin8','')
                data['sku']   = obj.get('sku','')
                imgs = obj.get('image',[])
                if isinstance(imgs, str): imgs = [imgs]
                data['images'] = [i if isinstance(i,str) else i.get('url','') for i in imgs][:3]
                brand = obj.get('brand',{})
                data['brand'] = brand.get('name','') if isinstance(brand,dict) else str(brand)
        except: pass
    
    # 2. Tabela atrybutów WooCommerce
    specs = []
    tbl = soup.find('table', class_=re.compile('woocommerce-product-attributes'))
    if tbl:
        for row in tbl.find_all('tr'):
            th = row.find('th'); td = row.find('td')
            if th and td:
                label = th.get_text(strip=True)
                value = td.get_text(' ', strip=True)
                if label and value:
                    specs.append({'label': label, 'value': value})
    data['specs'] = specs
    
    # 3. Jeśli brak EAN — szukaj w stronie
    if not data.get('ean'):
        m = re.search(r'(?:EAN|ean|Ean)[:\s]+(\d{8,13})', r.text)
        if m: data['ean'] = m.group(1)
    
    return data

# TEST na konkretnym produkcie
test_url = "https://www.bechcicki.pl/welna-fasadowa-rockwool-frontrock-max-e-15cm-id-p-0210521"
prod = scrape_product(test_url)
print(f"Nazwa: {prod.get('name','')}")
print(f"EAN:   {prod.get('ean','')}")
print(f"SKU:   {prod.get('sku','')}")
print(f"Marka: {prod.get('brand','')}")
print(f"Zdj:   {prod.get('images',[''])[0][:80]}")
print(f"Desc:  {str(prod.get('desc',''))[:120]}")
print(f"Specs ({len(prod.get('specs',[]))}):")
for s in prod.get('specs',[]):
    print(f"  {s['label']}: {s['value']}")


Nazwa: 
EAN:   
SKU:   
Marka: 
Zdj:   
Desc:  
Specs (0):
